# Home Credit · Loan amount with tabular foundation models
### TabTune on the Lexsi platform (SDK) · regression

Predict the **credit amount** a retail applicant should be offered (`AMT_CREDIT`) from their profile, bureau history and previous applications — a limit-setting and affordability use case.

| Step | What you will see |
|---|---|
| 1 | Connect, create a regression project |
| 2 | Data dashboard for the target and its drivers |
| 3 | **Leakage guard**: drop fields that restate the answer |
| 4 | Train several foundation models zero-shot, with SHAP and LIME |
| 5 | Model comparison from the platform's metrics |
| 6 | Explainability across models |
| 7 | Batch scoring on held-out applicants: predicted vs actual, error bands |
| 8 | One applicant, explained |
| 9 | Ensemble, distillation to a fast student, leaderboard |

**Before you start:** the processed Home Credit parquet files must be in Google Drive at `MyDrive/datasets/home-credit-default-risk/processed/`, and you need a Lexsi SDK access token.

## 1 · Connect

In [ ]:
# ⚙️ Install the Lexsi SDK (pinned to the version the backend was tested with)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lexsi-sdk==0.1.69"], check=True)
from importlib.metadata import version
print("lexsi-sdk", version("lexsi-sdk"))

In [ ]:
# 🔐 Authenticate (paste your Lexsi SDK access token when prompted)
from lexsi_sdk import lexsi
lexsi.login()

In [ ]:
# 🔧 Demo configuration — the only cell you should need to edit
import os
ORGANIZATION = "Lexsi Labs"
WORKSPACE    = "TabTunetesting"
PROJECT      = "TFM Demo · Home Credit regression"   # project_type is fixed per project, so regression gets its own
COMPUTE      = "T4.small"
UPLOAD_COMPUTE = "small"                      # CPU size for the default XGBoost build on first upload

DATA_DIR = os.environ.get("HC_DATA_DIR", "/content/drive/MyDrive/datasets/home-credit-default-risk/processed")
TARGET   = "AMT_CREDIT"
SAMPLE_TRAIN_N, SAMPLE_TEST_N = 4000, 1500          # the backend caps a build at 10,000 rows (train + test)

# Regression-capable roster. XRFM and TabICLv2 are commercially usable; Limix is research-licensed.
ROSTER = ["TabICLv2", "XRFM", "Limix", "TabDPT"]
RUN_SUPPORT_SET  = False    # support-set attribution is defined for classification only
RUN_ENSEMBLE     = True
RUN_DISTILLATION = True
RUN_LEADERBOARD  = True
LIME_ITERATIONS  = 200
TABPFN_TOKEN     = None     # set to add "TabPFNv3" to the roster

In [ ]:
# 🧰 Helpers: idempotent project setup, safe training, metric extraction, plotting
import time, json, re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from contextlib import contextmanager
from IPython.display import display, Markdown

BLUE, ORANGE, TEAL, GREY, DARK, RED = "#2F5BEA", "#B4541A", "#1B998B", "#9AA0A8", "#16181D", "#D7263D"
pd.set_option("display.max_columns", 50)


def get_or_create(workspace, name, project_type):
    """Reuse a project if it exists, otherwise create it. Safe to rerun."""
    try:
        project = workspace.project(name)
        print(f"reusing project: {name}")
    except Exception:
        project = workspace.create_project(name, modality="tabular", project_type=project_type)
        print(f"created project: {name}")
    return project


def upload_once(project, csv_path, tag, config=None, compute_type=None):
    """Upload a file unless this tag already has it; set the config only on the first upload.

    lexsi-sdk 0.1.69 bug: on the first upload (classic default model) upload_data() calls
    delete_file(uploaded_path) even when compute_type is valid, so the backend no longer has the
    file ("File path not found" / "No such file or directory"). delete_file is suspended for
    that one call only.
    """
    try:
        files = project.files()
        existing = set(files["file_name"]) if "file_name" in files else set()
    except Exception:
        existing = set()
    name = csv_path.split("/")[-1]
    has_config = project.config() != "Not Found"
    if name in existing and (has_config or config is None):
        print(f"already uploaded: {name} ({tag})")
        return
    if config is not None and not has_config:
        if name in existing:                      # left over from a failed first upload
            project.delete_file(name)
        cls = type(project)
        had_own, real = "delete_file" in cls.__dict__, cls.__dict__.get("delete_file")
        cls.delete_file = lambda self, *a, **k: "skipped"
        try:
            print(project.upload_data(csv_path, tag, config=config, compute_type=compute_type))
        finally:
            if had_own:
                cls.delete_file = real
            else:
                del cls.delete_file
    else:
        print(project.upload_data(csv_path, tag))


@contextmanager
def tolerant_drop():
    """project.models() drops columns that some backend builds do not return."""
    orig = pd.DataFrame.drop

    def safe(self, *a, **k):
        k.setdefault("errors", "ignore")
        return orig(self, *a, **k)

    pd.DataFrame.drop = safe
    try:
        yield
    finally:
        pd.DataFrame.drop = orig


RUNS = []          # one record per training call, shown in the run log


def trained_models(project):
    with tolerant_drop():
        df = project.models()
    return df


def summary_of(project, model_name):
    try:
        return project.model_summary(model_name=model_name)
    except Exception as exc:
        print(f"no summary for {model_name}: {exc}")
        return None


def gfi(summary, method):
    """Global feature importance dict {feature: value} for 'shap' or 'lime', or {}."""
    if summary is None:
        return {}
    block = (summary.model_results or {}).get("GFI", {}) or {}
    raw = block.get(f"{method}_gfi") or {}
    out = {}
    for k, v in raw.items():
        try:
            out[k] = float(np.mean(np.abs(list(v.values())))) if isinstance(v, dict) else float(v)
        except Exception:
            pass
    return out


def support_rows(summary):
    if summary is None:
        return pd.DataFrame()
    block = ((summary.model_results or {}).get("GFI", {}) or {}).get("support_set_attribution") or {}
    return pd.DataFrame(block.get("top_support_rows", []))


def top_bar(importance, title, n=15, color=BLUE):
    s = pd.Series(importance).abs().sort_values(ascending=False).head(n)[::-1]
    fig = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h", marker_color=color))
    fig.update_layout(template="plotly_white", height=36 * len(s) + 140, title=title,
                      xaxis_title="mean |attribution|", margin=dict(l=10, r=10, t=60, b=40))
    return fig

In [ ]:
def model_names(project):
    try:
        return set(trained_models(project)["model_name"].astype(str))
    except Exception:
        return set()


def train(project, label, **kwargs):
    """train_model with timing, a status line, and the exact name of the model it created."""
    before, t0 = model_names(project), time.time()
    try:
        project.train_model(**kwargs)
        status = "ok"
    except Exception as exc:
        status = f"failed: {str(exc)[:160]}"
        xai = (kwargs.get("data_config") or {}).get("xai_method", [])
        if "support_set" in xai and "not allowed" in str(exc):
            print(f"↺ {label}: support set is not enabled for this account; retrying with SHAP and LIME")
            kwargs["data_config"] = {**kwargs["data_config"], "xai_method": [m for m in xai if m != "support_set"]}
            return train(project, label, **kwargs)
    new = sorted(model_names(project) - before)
    RUNS.append({"run": label, "strategy": kwargs.get("training_strategy"), "model": str(kwargs.get("model")),
                 "created": ", ".join(new), "status": status, "minutes": round((time.time() - t0) / 60, 1)})
    print(("✅ " if status == "ok" else "❌ ") + f"{label} → {', '.join(new) or '-'}: {status} ({RUNS[-1]['minutes']} min)")
    return new[-1] if (status == "ok" and new) else None


def base_data_config(xai):
    return {"tags": ["Training"], "test_tags": ["Test"], "feature_exclude": FEATURE_EXCLUDE,
            "xai_method": xai, "lime_explainability_iterations": LIME_ITERATIONS}


def model_config_for(models):
    models = [models] if isinstance(models, str) else list(models)
    return {"tabpfn_token": TABPFN_TOKEN} if "TabPFNv3" in models and TABPFN_TOKEN else None


def eval_metrics(summary):
    return dict((summary.model_results or {}).get("evalution_metrics", {}) or {}) if summary else {}


def modelling_info(summary):
    return dict((summary.model_results or {}).get("modelling_info", {}) or {}) if summary else {}


def features_used(summary):
    return list((summary.model_results or {}).get("feature_actual_used", []) or []) if summary else []


def ensure_inference(project, model_name):
    """Make `model_name` the active model and enable it for inference."""
    project.activate_model(model_name)
    try:
        project.update_inference_model_status(model_name, True)
    except Exception:
        pass


def case_attributions(case, kind="shap", positive="1"):
    """Signed per-feature attribution dict from a case, picking the positive class when nested."""
    raw = getattr(case, f"{kind}_feature_importance", {}) or {}
    if raw and isinstance(next(iter(raw.values())), dict):
        key = positive if positive in raw else list(raw)[-1]
        raw = raw[key]
    out = {}
    for k, v in raw.items():
        try:
            out[k] = float(v)
        except Exception:
            pass
    return out


def signed_bar(attr, title, n=12):
    s = pd.Series(attr)
    s = s.reindex(s.abs().sort_values(ascending=False).index).head(n)[::-1]
    fig = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h",
                           marker_color=[RED if v > 0 else BLUE for v in s.values]))
    fig.update_layout(template="plotly_white", height=34 * len(s) + 150, title=title,
                      xaxis_title="contribution (red pushes the prediction up)", margin=dict(l=10, r=10, t=60, b=40))
    return fig

In [ ]:
# 📐 Read the platform's inference output and score it locally
from sklearn import metrics as skm


def prediction_columns(result, uploaded):
    """Columns the platform added to the uploaded tag (prediction, probabilities, ...)."""
    return [c for c in result.columns if c not in set(uploaded.columns)]


def positive_proba(result, added, positive="1"):
    """Best guess at the positive-class probability column among the added columns."""
    numeric = [c for c in added if pd.api.types.is_numeric_dtype(result[c])
               and result[c].between(0, 1).all() and result[c].nunique() > 2]
    ranked = sorted(numeric, key=lambda c: (str(positive) not in c.split("_")[-1] and not c.endswith(str(positive)),
                                            "prob" not in c.lower() and "score" not in c.lower()))
    return ranked[0] if ranked else None


def predicted_label(result, added):
    cands = [c for c in added if result[c].nunique() <= 20]
    named = [c for c in cands if "pred" in c.lower()]
    return (named or cands or [None])[0]

In [ ]:
def regression_dashboard(y_true, y_pred, title, unit=""):
    """Predicted vs actual, residuals, error by decile and residual distribution."""
    y, f = np.asarray(y_true, float), np.asarray(y_pred, float)
    res = y - f
    out = {"rmse": float(np.sqrt(np.mean(res ** 2))), "mae": float(np.mean(np.abs(res))),
           "r2": float(skm.r2_score(y, f)), "mape_%": float(np.mean(np.abs(res) / np.maximum(np.abs(y), 1e-9)) * 100)}
    dec = pd.qcut(y, 10, labels=False, duplicates="drop")
    by_dec = pd.DataFrame({"dec": dec, "ape": np.abs(res) / np.maximum(np.abs(y), 1e-9) * 100}).groupby("dec")["ape"].median()
    lo, hi = np.percentile(np.r_[y, f], [1, 99])
    fig = make_subplots(rows=2, cols=2, vertical_spacing=0.15,
                        subplot_titles=(f"Predicted vs actual · R² {out['r2']:.3f}", "Residuals vs prediction",
                                        "Median absolute % error by actual-value decile", "Residual distribution"))
    fig.add_trace(go.Scattergl(x=y, y=f, mode="markers", marker=dict(color=BLUE, size=5, opacity=0.45), name="loan"), 1, 1)
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], line=dict(color=DARK, dash="dot"), name="perfect"), 1, 1)
    fig.add_trace(go.Scattergl(x=f, y=res, mode="markers", marker=dict(color=ORANGE, size=5, opacity=0.45), showlegend=False), 1, 2)
    fig.add_hline(y=0, line_color=DARK, line_dash="dot", row=1, col=2)
    fig.add_trace(go.Bar(x=[f"D{int(d) + 1}" for d in by_dec.index], y=by_dec.values, marker_color=TEAL, showlegend=False), 2, 1)
    fig.add_trace(go.Histogram(x=res, nbinsx=50, marker_color=GREY, showlegend=False), 2, 2)
    fig.update_xaxes(title_text=f"actual {unit}", row=1, col=1); fig.update_yaxes(title_text=f"predicted {unit}", row=1, col=1)
    fig.update_xaxes(title_text=f"predicted {unit}", row=1, col=2); fig.update_yaxes(title_text="% error", row=2, col=1)
    fig.update_layout(template="plotly_white", height=760, title=title, legend=dict(orientation="h", y=-0.08))
    fig.show()
    return out

In [ ]:
def preflight_high_cardinality(df, cols, limit=0.95, digits=(4, 3, 2)):
    """The backend drops any column with more than 95% unique values (model.py:494-500), which also
    removes continuous features. Round those columns to fewer significant digits so they survive."""
    report = []
    for c in cols:
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue
        ratio = df[c].nunique() / len(df)
        if ratio <= limit:
            continue
        fixed = None
        for d in digits:
            rounded = df[c].map(lambda v: float(f"{v:.{d}g}") if pd.notna(v) else v)
            if rounded.nunique() / len(df) <= limit:
                fixed = d
                break
        report.append({"feature": c, "unique_ratio": round(ratio, 3),
                       "fix": f"rounded to {fixed} significant digits" if fixed else "dropped by backend"})
        if fixed:
            df[c] = df[c].map(lambda v: float(f"{v:.{fixed}g}") if pd.notna(v) else v)
    return pd.DataFrame(report)

In [ ]:
# 📚 Model catalogue as enforced by the Lexsi backend (xai/core/config.py, xai/utils/checks.py)
CATALOGUE = pd.DataFrame([
    # model,          classification, regression, support_set, shap_regression, commercial,  note
    ("OrionMoE",      True,  False, False, False, "yes",        "Lexsi mixture-of-experts, inference only"),
    ("OrionMSP",      True,  False, True,  False, "yes",        "Lexsi multi-scale sparse attention"),
    ("OrionBix",      True,  False, True,  False, "yes",        "Lexsi bi-axial attention"),
    ("OrionMSPv1.5",  True,  False, False, False, "yes",        "Lexsi"),
    ("TabICL",        True,  False, True,  False, "yes",        ""),
    ("TabICLv2",      True,  True,  False, True,  "yes",        ""),
    ("TabPFN",        True,  True,  False, True,  "unverified", ""),
    ("TabPFNv26",     True,  True,  True,  True,  "unverified", ""),
    ("TabPFNv3",      True,  True,  True,  True,  "no",         "needs a tabpfn_token"),
    ("TabDPT",        True,  True,  False, True,  "unverified", ""),
    ("Mitra",         True,  True,  False, False, "yes",        "no predict_proba"),
    ("ContextTab",    True,  True,  False, False, "no",         ""),
    ("Limix",         True,  True,  True,  True,  "no",         ""),
    ("TabFM",         True,  True,  False, False, "no",         "Google"),
    ("EXAONETabular", True,  False, False, False, "no",         "LG AI Research"),
    ("ILTM",          True,  True,  False, False, "yes",        ""),
    ("XRFM",          True,  True,  False, True,  "yes",        "trains from scratch"),
], columns=["model", "classification", "regression", "support_set", "shap_regression", "commercial", "note"]).set_index("model")


def xai_for(model, task):
    """Explainability methods the backend accepts for a simple build of `model`."""
    cap = CATALOGUE.loc[model]
    if task == "regression":
        return ["shap", "lime"] if cap.shap_regression else ["lime"]
    methods = ["shap", "lime"]
    if cap.support_set and RUN_SUPPORT_SET:
        methods.append("support_set")
    return methods

In [ ]:
# 🗂️ Organization → workspace → regression project
organization = lexsi.organization(ORGANIZATION)
workspace = organization.workspace(WORKSPACE)
project = get_or_create(workspace, PROJECT, "regression")
if TABPFN_TOKEN and "TabPFNv3" not in ROSTER:
    ROSTER.append("TabPFNv3")
assert all(CATALOGUE.loc[m, "regression"] for m in ROSTER), "every roster model must support regression"
display(CATALOGUE.loc[ROSTER].style.set_caption("Models in this demo (from the backend catalogue)"))

## 2 · Home Credit data

In [ ]:
# 📥 Load the processed Home Credit parquet files from Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

UID = "SK_ID_CURR"
X_train = pd.read_parquet(os.path.join(DATA_DIR, "X_train.parquet"))
X_val   = pd.read_parquet(os.path.join(DATA_DIR, "X_val.parquet"))
# The default label (TARGET) is an outcome observed after the loan: never a feature for loan sizing.
for _df in (X_train, X_val):
    _df.drop(columns=["TARGET"], inplace=True, errors="ignore")
    _df[TARGET] = pd.to_numeric(_df[TARGET], errors="coerce")
X_train, X_val = X_train[X_train[TARGET].notna()], X_val[X_val[TARGET].notna()]

bad = [c for c in X_train.columns if re.search(r'[<>\[\]",]', str(c)) or str(c) != str(c).strip()]
X_train.drop(columns=bad, inplace=True, errors="ignore"); X_val.drop(columns=bad, inplace=True, errors="ignore")
print(f"train {X_train.shape} · validation {X_val.shape}")
display(X_train[TARGET].describe().to_frame().T.style.format("{:,.0f}"))

In [ ]:
# 📊 Target and driver dashboard
smp = X_train.sample(min(20000, len(X_train)), random_state=0)
fig = make_subplots(rows=1, cols=3, subplot_titles=("Credit amount distribution", "Credit vs income (log scale)",
                                                     "Median credit by contract type"))
fig.add_trace(go.Histogram(x=smp[TARGET], nbinsx=60, marker_color=BLUE, showlegend=False), 1, 1)
if "AMT_INCOME_TOTAL" in smp:
    fig.add_trace(go.Scattergl(x=smp["AMT_INCOME_TOTAL"], y=smp[TARGET], mode="markers",
                               marker=dict(size=3, color=ORANGE, opacity=0.3), showlegend=False), 1, 2)
    fig.update_xaxes(type="log", row=1, col=2); fig.update_yaxes(type="log", row=1, col=2)
if "NAME_CONTRACT_TYPE" in smp:
    med = smp.groupby("NAME_CONTRACT_TYPE")[TARGET].median()
    fig.add_trace(go.Bar(x=med.index.astype(str), y=med.values, marker_color=TEAL, showlegend=False), 1, 3)
fig.update_layout(template="plotly_white", height=430, title="Home Credit · what drives the loan amount?")
fig.show()

## 3 · Leakage guard

Some fields restate the answer: `AMT_GOODS_PRICE` is the price of the goods the loan pays for, and `AMT_ANNUITY` is computed from the credit amount. Left in, they give a near-perfect R² that means nothing. We drop them, any engineered feature built from them, and anything correlated above 0.95 with the target.

In [ ]:
# 🛡️ Find and exclude fields that leak the target
numeric = X_train.select_dtypes("number").drop(columns=[UID, TARGET], errors="ignore")
corr = numeric.sample(min(50000, len(numeric)), random_state=0).corrwith(X_train[TARGET]).abs().sort_values(ascending=False)
explicit = [c for c in ("AMT_GOODS_PRICE", "AMT_ANNUITY") if c in X_train]
derived = [c for c in X_train.columns if str(c).startswith("FE_") and re.search("CREDIT|ANNUITY|GOODS", str(c))]
LEAKY = sorted(set(explicit + derived + corr[corr > 0.95].index.tolist()))

top = corr.head(15)[::-1]
fig = go.Figure(go.Bar(x=top.values, y=top.index, orientation="h",
                       marker_color=[RED if c in LEAKY else GREY for c in top.index]))
fig.add_vline(x=0.95, line_dash="dot", line_color=DARK)
fig.update_layout(template="plotly_white", height=520, title="Correlation with AMT_CREDIT · red = excluded as leakage",
                  xaxis_title="|correlation|")
fig.show()
FEATURE_EXCLUDE = LEAKY
FEATURES = [c for c in X_train.columns if c not in (UID, TARGET, *FEATURE_EXCLUDE)]
print(f"excluded {len(LEAKY)}: {LEAKY}")
print(f"{len(FEATURES)} features remain")

In [ ]:
# ✂️ Sample, backend pre-flight check, CSVs
train_sample = X_train.sample(min(SAMPLE_TRAIN_N, len(X_train)), random_state=42).reset_index(drop=True)
test_sample  = X_val.sample(min(SAMPLE_TEST_N, len(X_val)), random_state=42).reset_index(drop=True)
both = pd.concat([train_sample, test_sample])
fixes = preflight_high_cardinality(both, FEATURES)
train_sample, test_sample = both.iloc[:len(train_sample)].copy(), both.iloc[len(train_sample):].copy()
display(fixes if len(fixes) else Markdown("No feature would be dropped by the backend's 95%-unique rule."))
assert train_sample[TARGET].nunique() >= 100, "the backend needs at least 100 distinct target values"

TRAIN_CSV, TEST_CSV = "/content/hc_reg_demo_train.csv", "/content/hc_reg_demo_test.csv"
train_sample.to_csv(TRAIN_CSV, index=False)
test_sample.to_csv(TEST_CSV, index=False)
print(f"train sample {train_sample.shape} · test sample {test_sample.shape}")

In [ ]:
# ⬆️ Upload once
config = {"unique_identifier": UID, "true_label": TARGET, "pred_label": "",
          "feature_exclude": FEATURE_EXCLUDE, "drop_duplicate_uid": True, "handle_errors": True}
upload_once(project, TRAIN_CSV, "Training", config=config, compute_type=UPLOAD_COMPUTE)
upload_once(project, TEST_CSV, "Test")
display(project.files()[["file_name", "tag", "status", "created_at"]])
print("tags:", project.tags())

## 4 · Zero-shot training

Every model runs with `tunning_strategy="inference"`, requests the explanations the backend accepts for regression, and is scored by the platform on the **Test** tag. The leakage exclusions are passed on every build.

In [ ]:
# 🚀 One simple build per model
BUILDS = {}
for model in ROSTER:
    name = train(project, f"{model} · zero-shot", model=model, training_strategy="simple",
                 compute_type=COMPUTE, tunning_strategy="inference",
                 data_config=base_data_config(xai_for(model, "regression")), model_config=model_config_for(model))
    if name:
        BUILDS[model] = name
print("\nbuilt:", BUILDS)

## 5 · Model comparison

In [ ]:
# 🏁 Platform metrics side by side
SUMMARIES = {m: summary_of(project, n) for m, n in BUILDS.items()}
rows = []
for m, s in SUMMARIES.items():
    em = eval_metrics(s)
    rows.append({"model": m, "build": BUILDS[m], "R2": em.get("r2_score"), "RMSE": em.get("RootMeanSquaredError"),
                 "MAE": em.get("Mean_Absolute_Error"), "MAPE": em.get("Mean_Absolute_Per_Error"),
                 "features used": len(features_used(s)), "commercial": CATALOGUE.loc[m, "commercial"]})
board = pd.DataFrame(rows).set_index("model").sort_values("R2", ascending=False)
display(board.style.format({"R2": "{:.3f}", "RMSE": "{:,.0f}", "MAE": "{:,.0f}", "MAPE": "{:.3f}"}, na_rep="-")
        .background_gradient(subset=["R2"], cmap="Blues").set_caption("Scored by the platform on the Test tag"))
fig = make_subplots(rows=1, cols=2, subplot_titles=("R² (higher is better)", "MAE (lower is better)"))
fig.add_trace(go.Bar(x=board.index, y=board["R2"], marker_color=BLUE, text=board["R2"], texttemplate="%{text:.3f}"), 1, 1)
fig.add_trace(go.Bar(x=board.index, y=board["MAE"], marker_color=ORANGE, text=board["MAE"], texttemplate="%{text:,.0f}"), 1, 2)
fig.update_layout(template="plotly_white", height=430, showlegend=False,
                  title="Zero-shot foundation models · held-out applicants")
fig.show()
BEST = board["R2"].astype(float).fillna(-1e9).idxmax()
display(Markdown(f"**Best by R²: `{BEST}` ({BUILDS[BEST]})**"))
diag = modelling_info(SUMMARIES[BEST]).get("regression_diagnostics")
if diag:
    display(Markdown("**Regression diagnostics stored by the platform** (intervals, interval calibration, residuals)"))
    display(pd.json_normalize(diag).T.head(30))

## 6 · Explainability across models

In [ ]:
# 🔍 Global SHAP per model, cross-model heatmap and SHAP vs LIME
shap = {m: gfi(s, "shap") for m, s in SUMMARIES.items() if gfi(s, "shap")}
lime = {m: gfi(s, "lime") for m, s in SUMMARIES.items() if gfi(s, "lime")}
if shap:
    top_bar(shap.get(BEST) or next(iter(shap.values())), f"Global SHAP · {BEST}").show()
    mat = pd.DataFrame(shap).fillna(0)
    mat = mat.div(mat.max())
    top = mat.mean(axis=1).sort_values(ascending=False).head(15).index
    fig = px.imshow(mat.loc[top], color_continuous_scale="Blues", aspect="auto", text_auto=".2f",
                    title="Relative SHAP importance · top 15 features across models")
    fig.update_layout(height=620, template="plotly_white"); fig.show()
if BEST in shap and BEST in lime:
    b = pd.DataFrame({"SHAP": pd.Series(shap[BEST]), "LIME": pd.Series(lime[BEST])}).fillna(0)
    b = b.div(b.max())
    t = b.max(axis=1).sort_values(ascending=False).head(12).index[::-1]
    fig = go.Figure([go.Bar(y=t, x=b.loc[t, "SHAP"], name="SHAP", orientation="h", marker_color=BLUE),
                     go.Bar(y=t, x=b.loc[t, "LIME"], name="LIME", orientation="h", marker_color=ORANGE)])
    fig.update_layout(template="plotly_white", barmode="group", height=560, title=f"SHAP vs LIME · {BEST}")
    fig.show()

## 7 · Batch scoring on held-out applicants

In [ ]:
# 📦 Score the Test tag with the best model, evaluate locally
ensure_inference(project, BUILDS[BEST])
scored = project.model_inference(tag="Test", model_name=BUILDS[BEST], compute_type=COMPUTE)
scored = scored.merge(test_sample[[UID, TARGET]].rename(columns={TARGET: "actual"}), on=UID, how="left")
pred, actual = scored["Predicted_value_AutoML"].astype(float), scored["actual"].astype(float)
local = regression_dashboard(actual, pred, f"{BUILDS[BEST]} · {len(scored):,} held-out applicants", unit="(currency)")
display(pd.Series(local, name="local evaluation").to_frame().style.format("{:,.3f}"))

In [ ]:
# 💼 Business view: how often is the recommended amount close to the approved amount?
ape = (pred - actual).abs() / actual.clip(lower=1) * 100
bands = pd.cut(ape, [0, 10, 20, 35, 50, np.inf], labels=["within 10%", "10–20%", "20–35%", "35–50%", "over 50%"])
share = bands.value_counts(normalize=True).reindex(bands.cat.categories).mul(100)
fig = go.Figure(go.Bar(x=share.index.astype(str), y=share.values, marker_color=[TEAL, BLUE, GREY, ORANGE, RED],
                       text=share.values, texttemplate="%{text:.0f}%"))
fig.update_layout(template="plotly_white", height=420, yaxis_title="% of applicants",
                  title=f"{share.iloc[:2].sum():.0f}% of recommendations land within 20% of the approved amount")
fig.show()

## 8 · One applicant, explained

In [ ]:
# 🧑‍💼 Explain the recommendation for a held-out applicant with a large approved amount
CASE_UID = str(scored.sort_values("actual", ascending=False)[UID].iloc[len(scored) // 20])
case = project.case_predict(unique_identifier=CASE_UID, tag="Test", model_name=BUILDS[BEST], xai=["shap", "lime"])
display(case.inference_output())
sh, li = case_attributions(case, "shap"), case_attributions(case, "lime")
if sh:
    signed_bar(sh, f"SHAP · applicant {CASE_UID} · red raises the recommended amount").show()
if li:
    signed_bar(li, f"LIME · applicant {CASE_UID}").show()
top = pd.Series(sh or li).abs().sort_values(ascending=False).head(6).index
row = test_sample.assign(_k=test_sample[UID].astype(str)).set_index("_k").loc[CASE_UID]
display(pd.DataFrame({"applicant": row[top], "portfolio median": train_sample[top].median(numeric_only=True)})
        .style.set_caption("The applicant's top drivers next to the portfolio median"))

## 9 · Advanced builds

* **Ensemble** — greedy selection over the zero-shot models, optimising RMSE (explained with LIME).
* **Distillation** — the best model's knowledge compressed into a fast LightGBM student for real-time scoring.
* **Leaderboard** — the platform races the models on an internal hold-out by R² and refits the winner.

In [ ]:
# 🤝 Ensemble · 🧪 Distillation · 🏆 Leaderboard
ADV = {}
pool = list(BUILDS)[:3]
if RUN_ENSEMBLE and len(pool) >= 2:
    ADV["Ensemble"] = train(project, "Ensemble · greedy RMSE", model=pool, training_strategy="ensemble",
                            compute_type=COMPUTE, tunning_strategy="inference",
                            data_config=base_data_config(["lime"]), model_config=model_config_for(pool),
                            ensemble_config={"ensemble_strategy": "greedy_selection", "metric": "rmse",
                                             "weight_scheme": "performance", "cv_folds": 5,
                                             "holdout_fraction": 0.2, "greedy_ensemble_size": 50})
if RUN_DISTILLATION:
    ADV["Distilled"] = train(project, f"Distillation · {BEST} → LightGBM", model=BEST, training_strategy="distillation",
                             compute_type=COMPUTE, data_config=base_data_config(["shap", "lime"]),
                             model_config=model_config_for(BEST),
                             distillation_config={"student": "lgbm", "gbdt_backend": "lgbm", "temperature": 3.0,
                                                  "alpha": 0.7, "n_folds": 5, "adaptive_temperature": True,
                                                  "confidence_weighting": True})
if RUN_LEADERBOARD and len(BUILDS) >= 2:
    ADV["Leaderboard"] = train(project, "Leaderboard · R²", model=list(BUILDS), training_strategy="leaderboard",
                               compute_type=COMPUTE, tunning_strategy="inference",
                               data_config=base_data_config(["lime"]), model_config=model_config_for(list(BUILDS)),
                               leaderboard_config={"rank_by": "r2_score"})
ADV = {k: v for k, v in ADV.items() if v}
print(ADV)

In [ ]:
# 📊 Everything built today, on the same Test tag
rows = []
for label, name in {**BUILDS, **ADV}.items():
    s = summary_of(project, name)
    em = eval_metrics(s)
    rows.append({"build": label, "model": name, "R2": em.get("r2_score"), "RMSE": em.get("RootMeanSquaredError"),
                 "MAE": em.get("Mean_Absolute_Error")})
    mi = modelling_info(s)
    if label == "Leaderboard" and mi.get("leaderboard"):
        display(Markdown(f"**Leaderboard winner: `{mi.get('winner_model_name')}`**"))
        display(pd.DataFrame(mi["leaderboard"]))
final = pd.DataFrame(rows).set_index("build")
display(final.style.format({"R2": "{:.3f}", "RMSE": "{:,.0f}", "MAE": "{:,.0f}"}, na_rep="-")
        .background_gradient(subset=["R2"], cmap="Blues"))
fig = px.bar(final.reset_index(), x="build", y="R2", color="build", text_auto=".3f",
             title="R² on held-out applicants · zero-shot models vs advanced builds")
fig.update_layout(template="plotly_white", height=430, showlegend=False); fig.show()

In [ ]:
# 🧾 Run log for this session
display(pd.DataFrame(RUNS))
with tolerant_drop():
    display(project.models()[["model_name", "status", "tabular_model_type", "instance_type", "created_at"]]
            .sort_values("created_at", ascending=False).head(20))

## Summary

| Capability | Where it appeared |
|---|---|
| Leakage guard before any training | section 3 |
| Zero-shot regression with foundation models | section 4 |
| Platform metrics and regression diagnostics | section 5 |
| Global SHAP and LIME, cross-model view | section 6 |
| Batch scoring: predicted vs actual, error bands | section 7 |
| Per-applicant explanation | section 8 |
| Ensemble, distillation to a fast student, leaderboard | section 9 |

**Notes for the presenter**
* An R² near 1.0 on this data means a leak, not a win: section 3 removes the fields that restate the loan amount.
* SHAP for regression is available for TabICLv2, XRFM, Limix, TabDPT, TabPFN v2.6/v3; Mitra, ContextTab, TabFM and ILTM are explained with LIME only.
* Support-set attribution is a classification feature; see the Lending Club notebook.